<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Workflow Runner

Chain analytic extractors using YAML-defined workflows directly from `WorkflowManager`.

The runner now supports:

- **Per-step `input_from`** — explicitly select the entity_list source: `"original"`, another step's name, or omit to default to the first `depends_on` (else `"original"`).
- **`transform` blocks** — reshape the entity list before extraction. A step with only a `transform` (no `extractor`) stores the transform output as its `results_df`.
- **Conditions with `depends_on`** — filter entities by upstream results (`eq`, `ne`, `in`, `notnull`).
- **Parallel execution per level** — independent steps in the same topological level run in parallel via `ThreadPoolExecutor` (configurable with `max_parallel_steps`).
- **Plotly DAG visualization** — `manager.visualize_workflow()` returns a `plotly.graph_objects.Figure`.

**Workflow example:** Emergence → Greenness + Disease (parallel)
- Step 1: Compute emergence for all entities
- Step 2: For entities with confirmed emergence, run greenness detection
- Step 3: For entities with confirmed emergence, run disease risk (parallel with step 2)

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import yaml
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Load entities**

### Option 1 - Load entities from EarthDaily platform

In [ ]:
manager.load_seasonfields(sowing_date_gte="2023-04-01", crop_id="CORN")

print(f"\n🔎 Loaded {len(manager.sfd_list)} entities")
print(manager.sfd_list[['id', 'name']].head())
print(manager.sfd_list.columns)

In [ ]:
#create a subset for testing
manager.sfd_list= manager.sfd_list.head(10)

## **📥 Step 3: Load & inspect the workflow**

Workflows are loaded directly through the `WorkflowManager`. The previous `PrefectWorkflowRunner` class has been folded into the manager:

- `manager.load_workflow(path)` — parse + validate YAML (raises on circular deps, unknown `depends_on`/`input_from`, duplicate step names).
- `manager.inspect_workflow()` — programmatic dict summary (steps, execution levels, transforms, classifications).
- `manager.visualize_workflow()` — interactive Plotly DAG.

In [ ]:
workflow_path = "workflow/emergence_greenness_disease.yml"

manager.load_workflow(workflow_path)

# Programmatic summary (steps, execution levels, transform/extractor classification)
info = manager.inspect_workflow()
print(f"Workflow: {info['name']}")
print(f"Levels: {info['execution_levels']}")
for s in info["steps"]:
    print(f"  - {s['name']}: input_from={s['input_from']}, "
          f"transform={s.get('transform') or '-'}, extractor={s.get('extractor') or '-'}")

In [ ]:
# visualize_workflow() returns a Plotly Figure — call .show() (or leave as last expression)
manager.visualize_workflow().show()

In [ ]:
# visualize_workflow() returns a Plotly Figure — call .show() (or leave as last expression)
manager.interactive_workflow()

## **▶️ Step 4: Load & run the workflow**

In [ ]:
# Run on all loaded entities (or pass a subset via entity_list=...)
results = manager.run_workflow()

## **📊 Step 5: Explore results**

### Overall summary

In [ ]:
import pandas as pd

summary_rows = []
for step_name, res in results.items():
    df = res.get("results_df", pd.DataFrame())
    summary_rows.append({
        "step": step_name,
        "skipped": res.get("skipped", False),
        "result_rows": len(df),
        "errors": len(res.get("global_errors", [])),
        "failed_ids": len(res.get("failed_ids", [])),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

### Emergence results

In [ ]:
emergence_df = results["emergence"]["results_df"]
print(f"🌱 Emergence: {len(emergence_df)} rows")
print(f"   Columns: {list(emergence_df.columns)}")

if not emergence_df.empty:
    print(f"\n📊 Emergence status distribution:")
    if 'emergence_status' in emergence_df.columns:
        print(emergence_df['emergence_status'].value_counts())
    if 'confirmation_status' in emergence_df.columns:
        print(f"\n📊 Confirmation status distribution:")
        print(emergence_df['confirmation_status'].value_counts())
    display(emergence_df.head(10))

### Greenness results

In [ ]:
greenness_df = results["greenness"]["results_df"]
print(f"🌿 Greenness: {len(greenness_df)} rows")

if not greenness_df.empty:
    print(f"   Columns: {list(greenness_df.columns)}")
    display(greenness_df.head(10))
else:
    print("   ⚠️ No entities passed the emergence condition for greenness")

### Disease results

In [ ]:
disease_df = results["disease"]["results_df"]
print(f"🦠 Disease: {len(disease_df)} rows")

if not disease_df.empty:
    print(f"   Columns: {list(disease_df.columns)}")
    display(disease_df.head(10))
else:
    print("   ⚠️ No entities passed the emergence condition for disease")

### Error inspection

In [ ]:
for step_name, res in results.items():
    errors = res.get("global_errors", [])
    failed = res.get("failed_ids", [])
    if res.get("skipped"):
        print(f"⏭️  {step_name}: skipped (no entities after input_from / transform / condition)")
        continue
    if errors:
        print(f"\n❌ {step_name}: {len(errors)} errors, {len(failed)} failed IDs")
        for i, err in enumerate(errors[:3]):
            print(f"   Error {i+1}: {err}")
        if len(errors) > 3:
            print(f"   ... and {len(errors) - 3} more")
    else:
        print(f"✅ {step_name}: No errors")

### DAG result visualization

`manager.visualize_workflow_results()` renders the same DAG layout as `visualize_workflow()` but colors each node by run outcome (success / partial / failed / skipped / not run) and annotates it with row + error counts. Hover for per-step details (failed_ids sample, skip reason, cache stats).

Requires a prior `run_workflow()` call — it reads from `manager.workflow_results`.

In [ ]:
# Post-run DAG colored by outcome (rows / errors / skipped per step).
manager.visualize_workflow_results().show()

## **🧩 Step 6: Run on a subset of entities**

In [ ]:
# Run workflow on just the first 10 entities for quick testing
subset = manager.sfd_list.head(10)
print(f"🔬 Running workflow on {len(subset)} entities...\n")

results_subset = manager.run_workflow(entity_list=subset)

In [ ]:
for step_name, res in results_subset.items():
    df = res.get("results_df", pd.DataFrame())
    status = "skipped" if res.get("skipped") else f"{len(df)} result rows"
    errors = len(res.get("global_errors", []))
    print(f"{step_name}: {status}" + (f" ({errors} errors)" if errors else ""))

## **📝 Step 7: Create a custom workflow YAML**

Schema:

```yaml
workflow:
  name: "My Custom Workflow"
  description: "Description of what this workflow does"

  settings:                       # Global defaults inherited by all steps
    max_workers: 5                # parallelism within a single extractor
    partial_frequency: 50         # save partial results every N entities
    max_parallel_steps: 4         # parallelism across independent steps in the same level
    fail_safe: false              # if true, errors don't abort bulk extraction
    column_mapping:               # optional global column mapping
      id: my_id
      geometry: my_geom

  steps:
    - name: step_name             # Unique identifier
      extractor: ClassName        # e.g. EmergenceExtractor (omit for transform-only steps)
      module: utils.module        # Python import path
      depends_on: other_step      # Optional: schedule after this step (string or list)
      input_from: original        # Optional: "original" | "<step_name>"
                                  #   default = first depends_on, else "original"

      transform:                  # Optional: reshape entity_list before extraction
        module: my.transforms
        function: my_transform
        params: {key: value}

      condition:                  # Optional: filter entities from upstream results
        depends_on: other_step    # which upstream step's results_df to filter on
        column: col_name
        operator: eq              # eq | ne | in | notnull
        value: "target_value"

      setup:
        method: setup_*_parameters
        params: {key: value}

      run:
        method: process_*_bulk_*
        params:
          prefix: "step_name"
          skip_export: false
          generate_report: false  # optional HTML report
          fail_safe: false        # overrides settings.fail_safe
          max_workers: 10         # overrides settings.max_workers
```

**Step roles** (auto-classified by `inspect_workflow()` / `visualize_workflow()`):
- **Extractor only** — has `extractor` (+ `setup`, `run`).
- **Transform + Extractor** — has both `transform` and `extractor`.
- **Transform-only** — has only `transform`; output `DataFrame` is stored as the step's `results_df`.

**`input_from` resolution rules:**
1. Explicit value in YAML wins.
2. If omitted and step has `depends_on`, defaults to the first upstream step name.
3. Otherwise defaults to `"original"` (the `entity_list` passed to `run_workflow`).

Validated at `load_workflow()`: `input_from` must be `"original"` or a step in an earlier execution level.

**Available extractors:**
- `EmergenceExtractor` (`earthdaily.agriculture.processors.processor_emergence_functions`)
- `GreennessExtractor` (`earthdaily.agriculture.processors.processor_greenness_functions`)
- `DiseaseExtractor` (`earthdaily.agriculture.extractors.disease_function`)
- `HarvestExtractor` (`earthdaily.agriculture.processors.processor_harvest_functions`)
- `CoverageExtractor` (`earthdaily.agriculture.extractors.coverage_function`)
- `InSeasonMonitoringExtractor` (`earthdaily.agriculture.processors.processor_inseason_monitoring_functions`)
- `cropidExtractor` (`earthdaily.agriculture.processors.processor_cropid_functions`)
- `TillageExtractor` (`earthdaily.agriculture.processors.processor_tillage_functions`)
- `BaresoilExtractor` (`earthdaily.agriculture.processors.processor_baresoil_function`)
- `HistoricalScoreExtractor` / `InseasonScoreExtractor` (`earthdaily.agriculture.processors.processor_score_functions`)
- `MRTSExtractor` / `VegationTsExtractor` (`earthdaily.agriculture.extractors.VTS_functions`)
- `LrtsExtractor` (`earthdaily.agriculture.processors.processor_lrts_functions`)
- `WeatherExtractor` (`earthdaily.agriculture.extractors.weather_functions`)

**Condition operators:** `eq`, `ne`, `in`, `notnull`

In [ ]:
# Example: create a simple emergence-only workflow programmatically
import yaml

custom_workflow = {
    "workflow": {
        "name": "Emergence Only",
        "description": "Simple emergence extraction for testing",
        "settings": {
            "max_workers": 10,
            "partial_frequency": 25,
            # Global filename prefix; final files become
            # <output_prefix>_<step_prefix>_results_<datetime>_<suffix>.csv
            "output_prefix": "emergence_only_run",
            # Map platform-style columns (crop.id, sowingDate) to the canonical
            # names that EmergenceExtractor expects (crop, sowing_date).
            "column_mapping": {
                "crop": "crop.id",
                "sowing_date": "sowingDate",
            },
        },
        "steps": [
            {
                "name": "emergence",
                "extractor": "EmergenceExtractor",
                "module": "earthdaily.agriculture.processors.processor_emergence_functions",
                "setup": {
                    "method": "setup_emergence_parameters",
                    "params": {
                        "emergence_type": "INSEASON",
                        "season_duration": 120,
                        "season_start_day": 1,
                        "season_start_month": 4,
                        "year": 2025,
                        "data_source": "LR",
                    },
                },
                "run": {
                    "method": "process_emergence_bulk_extraction_parallel",
                    "params": {
                        "prefix": "emergence_test",
                        "skip_export": True,
                    },
                },
            }
        ],
    }
}

custom_path = "workflow/emergence_only.yml"
with open(custom_path, "w") as f:
    yaml.dump(custom_workflow, f, default_flow_style=False, sort_keys=False)

print(f"✅ Custom workflow saved to {custom_path}")

# Load and run via the manager
manager.load_workflow(custom_path)
manager.visualize_workflow().show()
custom_results = manager.run_workflow(entity_list=manager.sfd_list.head(5))

emergence_res = custom_results["emergence"]
print(
    f"\nResults: {len(emergence_res.get('results_df', pd.DataFrame()))} rows, "
    f"{len(emergence_res.get('global_errors', []))} errors"
)

## **🔁 Step 8: Transforms & `input_from`**

A `transform` block lets you reshape the entity list between steps — useful for projecting upstream `results_df` rows back into entity records, fanning out by date, joining metadata, etc.

A transform function has the signature:

```python
def my_transform(entity_list: pd.DataFrame, upstream_results: dict, params: dict) -> pd.DataFrame:
    ...
    return new_entity_list
```

`upstream_results` is the dict of all completed step results so far, and `params["_workflow_manager"]` gives access to auth/config if needed.

Common patterns:

- **Transform-only step** — omit `extractor`; the transform's output becomes the step's `results_df` (handy as a derived input feeder).
- **`input_from: "<step_name>"`** — feed a step the `results_df` of an earlier step (instead of the original entity list) without writing a transform.
- **`input_from: "original"`** — explicitly opt out of the auto-chain when a step has `depends_on` only for ordering.

In [ ]:
# Example: emergence -> coverage on the emergence dates
# Step 2 uses a transform to convert the emergence results_df into a coverage entity list.

example_yaml = {
    "workflow": {
        "name": "Emergence then Coverage",
        "settings": {"max_workers": 5, "partial_frequency": 25},
        "steps": [
            {
                "name": "emergence",
                "extractor": "EmergenceExtractor",
                "module": "earthdaily.agriculture.processors.processor_emergence_functions",
                "setup": {
                    "method": "setup_emergence_parameters",
                    "params": {
                        "emergence_type": "INSEASON",
                        "season_duration": 120,
                        "season_start_month": 4,
                        "year": 2025,
                        "data_source": "LR",
                    },
                },
                "run": {
                    "method": "process_emergence_bulk_extraction_parallel",
                    "params": {"prefix": "emergence", "skip_export": True},
                },
            },
            {
                "name": "coverage_around_emergence",
                "depends_on": "emergence",
                # input_from omitted -> defaults to "emergence" (its results_df)
                "transform": {
                    "module": "app.transforms",          # your project module
                    "function": "emergence_to_coverage", # turns emergence rows into coverage entities
                    "params": {"window_days": 14},
                },
                "extractor": "CoverageExtractor",
                "module": "earthdaily.agriculture.extractors.coverage_function",
                "setup": {
                    "method": "setup_coverage_parameters",
                    "params": {"vegetation_index": "NDVI", "clear_cover_min": 95},
                },
                "run": {
                    "method": "process_entity_coverage_bulk_parallel",
                    "params": {"prefix": "coverage", "skip_export": True},
                },
            },
        ],
    }
}

# This cell only prints the YAML — to run it, write a real `app.transforms.emergence_to_coverage`
# function that returns a DataFrame with the coverage entity columns (id, geometry, start_date, end_date).
print(yaml.dump(example_yaml, sort_keys=False))

## **Step 9 - run_prefix argument**

`run_workflow()` accepts an optional `run_prefix` that's inserted between
the workflow-level `settings.output_prefix` and the per-step prefix in
output filenames. Useful for re-running the same YAML with different tags
(per client / per date / per scenario) without mutating the loaded config.

Final filename pattern:
```
<output_prefix>_<run_prefix>_<step_prefix>_results_<timestamp>.csv
```
Any of the three layers can be absent. The `run_prefix` is reset to `None`
after each call so a later `run_workflow()` without the arg doesn't inherit it.

In [ ]:
# Re-run the loaded workflow with a per-run tag.
# Inspect the resulting filenames in `manager.partial_result_dir` after the run.
subset = manager.sfd_list.head(5)
results_a = manager.run_workflow(entity_list=subset, run_prefix='2026_05_07_clientA')
print(f'Run with prefix completed: {list(results_a.keys())}')
print(f'manager._run_prefix after run: {manager._run_prefix}  # should be None (reset)')

## **Step 10 - `enabled: false` step flag**

Each step accepts an optional `enabled: bool` field (default `true`).
When `false`, the step is loaded but not executed; the runner records
`{'skipped': True, 'reason': 'disabled', 'results_df': <empty>}` for it.
Downstream steps that `depends_on` (or `input_from`) the disabled step
cascade-skip via the existing empty-upstream branch.

Useful for temporarily turning off a step in source-controlled YAML
without commenting it out and re-wiring `depends_on` references.

In [ ]:
# Use the already-loaded multi-step workflow. Simulate `enabled: false` at the
# YAML level by patching step_map directly — load_workflow() does exactly this when
# it parses an `enabled: false` field. We restore the patch at the end so the rest
# of the notebook isn't affected.
manager.load_workflow(workflow_path)

# Pick the second step to disable (cascade-skips anything that depends on it).
target = manager.workflow_steps[1]["name"]
print(f"Disabling step: {target}")
manager.step_map[target]["enabled"] = False

# inspect_workflow surfaces the flag for every step
print("\nAfter patching, inspect_workflow() shows:")
for s in manager.inspect_workflow()["steps"]:
    print(f"  {s['name']:20s} enabled={s['enabled']}")

# Run — the disabled step (and anything depending on it) skip cleanly.
results = manager.run_workflow(entity_list=manager.sfd_list.head(3))
print()
for step, res in results.items():
    if res.get("skipped"):
        print(f"  {step:20s} skipped (reason: {res.get('reason') or 'cascade'})")
    else:
        print(f"  {step:20s} {len(res['results_df'])} rows")

# Restore so the rest of the notebook runs against the original YAML state.
del manager.step_map[target]["enabled"]


## **Step 11 - `disabled_steps` runtime kwarg**

Headless equivalent of the widgets: pass `disabled_steps=[...]` to
`run_workflow()` to skip steps for *this run only*. The override is
applied to `step_map[*]['enabled']` at the start of the run and rolled
back in a `finally` block - even if a step raises mid-run, the loaded
YAML state is preserved.

Useful for CLI / Argo / scheduled runs that want to skip steps without
editing the YAML on disk.

In [ ]:
# Reload the original multi-step workflow.
manager.load_workflow(workflow_path)

# Snapshot the loaded enabled state - must be unchanged after the run.
before = {s['name']: 'enabled' in manager.step_map[s['name']] for s in manager.workflow_steps}
print('Before:', {k: ('explicit' if v else 'absent') for k, v in before.items()})

# Run with one step disabled at runtime
results = manager.run_workflow(
    entity_list=manager.sfd_list.head(5),
    disabled_steps=['greenness'],   # change to a step name your YAML actually contains
)

after = {s['name']: 'enabled' in manager.step_map[s['name']] for s in manager.workflow_steps}
print('After: ', {k: ('explicit' if v else 'absent') for k, v in after.items()})
print('Override rolled back:', before == after)
print()
for step, res in results.items():
    if res.get('skipped'):
        status = f"skipped ({res.get('reason') or 'cascade'})"
    else:
        status = f"{len(res['results_df'])} rows"
    print(f"  {step:12s} {status}")


## **Step 12 - `select_steps_to_run()` checkbox widget**

Notebook UI for ad-hoc step selection. Returns an `ipywidgets.VBox`
with one checkbox per step plus a 'Run workflow' button. Toggling a
checkbox flips `step_map[name]['enabled']` in place; the button calls
`run_workflow()` with the chosen subset.

Per-step annotations show the extractor class and `depends_on` so you
can predict cascade-skip behaviour before clicking Run.

In [ ]:
# Render the widget — toggle checkboxes, then click 'Run workflow'.
# State persists on the manager so subsequent run_workflow() calls honor it.
manager.select_steps_to_run(entity_list=manager.sfd_list.head(5))

## **Step 13 - `interactive_workflow()` clickable DAG**

Same layout as `visualize_workflow()` but returns a `FigureWidget` you
can click. Clicking a node toggles its `enabled` state and recolors
it grey. State persists on the manager and is honored by the next
`run_workflow()` call.

Note: cascade-skip of downstream steps is enforced at runtime via the
empty-upstream branch — the widget colours only the directly-clicked
node, not the whole downstream subtree.

**Tip:** clicks only fire on the live widget. If you call `.show()`
you'll get a static figure — leave it as the bare last expression.

In [ ]:
# Click nodes to toggle, then run.
manager.interactive_workflow()

In [ ]:
# After toggling above, re-run honoring the chosen subset.
results = manager.run_workflow(entity_list=manager.sfd_list.head(5))
for step, res in results.items():
    if res.get('skipped'):
        status = f"skipped ({res.get('reason') or 'cascade'})"
    else:
        status = f"{len(res['results_df'])} rows"
    print(f"  {step:12s} {status}")


## **Step 14 — Dynamic date sentinels in YAML**

`load_workflow()` rewrites a small set of sentinel strings under `setup.params` and `entity_source.params` to ISO date strings **at load time**. Scheduled cron-driven workflows can use `today`, `yesterday`, `today-Nd/Nw/Nm`, and `today+N` (forecasts) without ever editing the YAML.

| Sentinel | Resolves to | Example |
|---|---|---|
| `today` | today's date in workflow TZ (UTC by default) | `2026-05-12` |
| `yesterday` | `today − 1 day` | `2026-05-11` |
| `today±N` | `today ± N days` (unit defaults to days) | `today+10` → `2026-05-22` |
| `today±Nd` / `today±Nw` / `today±Nm` | days / weeks / 30-day months | `today-2w` → `2026-04-28` |

See `docs/09b - Workflow_YAML_reference.md` § _Dynamic date sentinels_ for the full grammar. Settings can opt into a non-UTC zone with `settings.timezone: America/Chicago`.

In [ ]:
import yaml
from pathlib import Path

# Build a small workflow with sentinels in setup.params. The 'today',
# 'today-30d', and 'today+10' strings get rewritten at load_workflow() time.
sentinel_cfg = {
    'workflow': {
        'name': 'sentinel-demo',
        # Optional: route 'today' through a named IANA zone.
        # 'settings': {'timezone': 'America/Chicago'},
        'steps': [
            {
                'name': 'coverage_recent',
                'extractor': 'CoverageExtractor',
                'module': 'earthdaily.agriculture.extractors.coverage_function',
                'setup': {
                    'method': 'setup_coverage_parameters',
                    'params': {
                        'vegetation_index': 'NDVI',
                        'start_date': 'today-30d',   # rolling 30-day window
                        'end_date':   'today',       # always up to today
                    },
                },
                'run': {
                    'method': 'process_entity_coverage_bulk_parallel',
                    'params': {'prefix': 'coverage_recent', 'skip_export': True},
                },
            },
            {
                'name': 'weather_forecast',
                'extractor': 'WeatherExtractor',
                'module': 'earthdaily.agriculture.extractors.weather_functions',
                'setup': {
                    'method': 'setup_weather_parameters',
                    'params': {
                        'start_date': 'today',
                        'end_date':   'today+10',    # 10-day forecast window
                    },
                },
                'run': {
                    'method': 'process_entity_weather_bulk_parallel',
                    'params': {'prefix': 'weather_forecast', 'skip_export': True},
                },
            },
        ],
    }
}

sentinel_path = Path(manager.project_root) / 'workflow' / 'sentinel_demo.yml'
sentinel_path.parent.mkdir(parents=True, exist_ok=True)
sentinel_path.write_text(yaml.dump(sentinel_cfg))

manager.load_workflow(str(sentinel_path))

# After load, sentinels have been replaced with real ISO dates in place.
for step in manager.workflow_cfg['steps']:
    p = step['setup']['params']
    print(f"{step['name']:18s}  start={p.get('start_date')!r:14s}  end={p.get('end_date')!r}")

## **Step 15 — Storage paths: local, S3, env-var override**

Per `docs/13` Principle 1 ("paths are the mode"), routing writers to S3 is a single path change. Three ways to control where the writer surface lands:

1. **Local (default)** — `WorkflowManager("prod")`. Results go to `<project_root>/results`.
2. **Explicit constructor kwargs** — point any or all writers at `s3://...` (or `gs://`, `az://`, …) at construction time.
3. **`EDAGRO_OUTPUT_PREFIX` env var** — canonical for container deploys (see `docs/14`, Pattern B). Set the env var and `WorkflowManager` derives `output_result_dir` / `partial_result_dir` / `cache_dir` from it automatically.

**Precedence:** explicit kwargs > env var > setup-environment local defaults.

Pointing `cache_dir` at a remote path auto-disables caching with a one-time warning (atomic-rename has no S3 equivalent).

In [ ]:
# Current manager's resolved writer paths (local defaults in this dev notebook).
print(f'output_result_dir  = {manager.output_result_dir}')
print(f'partial_result_dir = {manager.partial_result_dir}')
print(f'cache_dir          = {manager.cache_dir}')

# --- Demo: build a fresh manager that would route every writer to S3. ---
# Uncomment to try (needs the [s3] extras installed and AWS creds available).
#
# from earthdaily.agriculture.services.workflow_manager import WorkflowManager
# s3_manager = WorkflowManager(
#     'prod',
#     log_to_console=True, log_level='INFO',
#     log_to_console_only=True,                                   # stdout-only — for containers
#     output_result_dir='s3://my-bucket/runs/2026-05-12/results',
#     partial_result_dir='s3://my-bucket/runs/2026-05-12/partials',
#     cache_dir='s3://my-bucket/runs/2026-05-12/cache',           # auto-disables cache
# )
# print(f's3 manager writes to: {s3_manager.output_result_dir}')

# --- Demo: EDAGRO_OUTPUT_PREFIX env var path (uncomment to try) ---
# import os
# os.environ['EDAGRO_OUTPUT_PREFIX'] = 's3://my-bucket/runs/2026-05-12/<workflow>'
# env_manager = WorkflowManager('prod', log_to_console=False)
# # env_manager.output_result_dir  == 's3://my-bucket/runs/2026-05-12/<workflow>/results'
# # env_manager.partial_result_dir == 's3://my-bucket/runs/2026-05-12/<workflow>/partials'
# # env_manager.cache_dir          == 's3://my-bucket/runs/2026-05-12/<workflow>/cache'
# del os.environ['EDAGRO_OUTPUT_PREFIX']

## **Step 16 — `publish_to_s3` transform (post-extraction local→S3 hand-off)**

For runs that wrote results locally and want to push everything to S3 *after the fact*, hook the built-in `publish_to_s3` transform onto the end of a workflow. The earlier steps land artifacts under `results/` (or wherever `output_result_dir` points); the publish step finds the most recent `*_manifest_*.json` and uploads every file it references via fsspec.

For runs that should write *directly* to S3 (during extraction), use the Step 15 kwargs — `publish_to_s3` is unnecessary in that path.

See `docs/09b § Built-in transform — publish_to_s3` for the full param list and behaviour.

In [ ]:
# Example YAML — a publish step that runs after the extractor and uploads
# every tracked file in the manifest to s3_prefix/...
publish_step = {
    'name': 'publish',
    'transform': {
        'module': 'earthdaily.agriculture.export.s3_publish',
        'function': 'publish_to_s3',
        'params': {
            's3_prefix': 's3://my-bucket/runs/2026-05-12',  # required
            'output_dir': 'results',                          # default 'results'
            's3_formats': ['parquet'],                        # optional format filter
            # 'storage_options': {'client_kwargs': {'endpoint_url': '...'}},  # MinIO / LocalStack
            # 'manifest_path': '<explicit-manifest.json>',     # override the glob discovery
        },
    },
}
import yaml
print(yaml.dump([publish_step], sort_keys=False))

# Programmatic call (outside of run_workflow):
# from earthdaily.agriculture.export import publish_local_run_to_s3
# patched_manifest_path = publish_local_run_to_s3(
#     manifest_path='results/coverage_manifest_2026-05-12.json',
#     s3_prefix='s3://my-bucket/runs/2026-05-12',
# )
# The manifest is rewritten in place with an s3_upload block per file and
# a top-level s3_publish summary — see manifest.print_manifest().

## **Step 17 — `WorkflowRunReporter` (`generate_report=True`)**

Per-run HTML + JSON receipt at the **workflow** level. Mirror of the per-extractor `ExtractionReporter` (`generate_report=True` on any bulk method) but one layer up — operates over the dict of step results that `run_workflow` returns.

Pass `generate_report=True` to opt in. The manager builds a `WorkflowRunReporter`, populates its context from the loaded YAML (workflow name, `output_prefix`, env, entity count), times the run, and stashes the configured reporter at `manager.last_run_reporter` so you can:

1. **Pull the receipt as a dict** — `manager.last_run_reporter.render_json()` for interactive inspection in the notebook.
2. **Amend + render the two-call API** — when the consumer builds the final per-field CSV downstream of `run_workflow`, call `reporter.amend_run_context(report_path=...)` before `render_html()` / `render_json(output_path=...)` so the path lands in both outputs.

`report_options` accepted keys (typos raise `ValueError` listing the accepted names):

| Bucket | Keys |
|---|---|
| **Reporter constructor** | `include_step_table`, `include_data_preview`, `preview_rows`, `max_errors_shown` |
| **Run-context fields** | `parameters`, `column_mapping`, `resolved_yaml` |

JSON receipt shape is `schema_version: 1` — `{workflow, entities, duration_seconds, report_path, steps[], errors[]}` — designed to feed Slack notifications, S3 archives, downstream dashboards without per-project shaping. See `docs/09 - Workflow_architecture.md` § _Run reporting_ for the full reference.

In [ ]:
# Re-load the multi-step workflow and re-run with generate_report=True.
import tempfile
from pathlib import Path

manager.load_workflow(workflow_path)

results = manager.run_workflow(
    entity_list=manager.sfd_list.head(5),
    generate_report=True,
    report_options={
        # Run-context: surfaces in the JSON receipt and the rendered HTML.
        "parameters": {"scenario": "notebook_demo", "limit": 5},
        # Constructor knob: cap error rows in the HTML table (JSON keeps all).
        "max_errors_shown": 10,
    },
)

reporter = manager.last_run_reporter
print(f"Reporter type: {type(reporter).__name__}")

# 1) Bare-dict receipt — no file write, useful for interactive inspection.
receipt = reporter.render_json()
print(f"schema_version : {receipt['schema_version']}")
print(f"workflow       : {receipt['workflow']}")
print(f"entities       : {receipt['entities']}")
print(f"duration_seconds: {receipt['duration_seconds']:.3f}")
print(f"steps          : {[s['name'] + ':' + s['kind'] for s in receipt['steps']]}")
print(f"errors         : {len(receipt['errors'])}")

In [ ]:
# 2) Two-call API — amend report_path AFTER the downstream consumer builds the
#    final per-field CSV, then render both HTML and JSON to disk. Mirrors the
#    <Client> daily-extraction pattern (see todo.md WorkflowRunReporter entry).

with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)

    # Pretend a downstream export step just produced the final report CSV.
    final_csv = tmp / "demo_final_report.csv"
    final_csv.write_text("entity_id\na\nb\nc\n")

    reporter.amend_run_context(report_path=str(final_csv))

    html_path = tmp / "run_report.html"
    json_path = tmp / "run_receipt.json"
    reporter.render_html(output_path=str(html_path))
    final_receipt = reporter.render_json(output_path=str(json_path))

    print(f"HTML report : {html_path.stat().st_size} bytes at {html_path}")
    print(f"JSON receipt: {json_path.stat().st_size} bytes at {json_path}")
    print(f"report_path : {final_receipt['report_path']}")

## **Step 18 — Parquet export via `settings.export_format`**

Results export as **CSV by default**; setting `settings.export_format: parquet` in the
workflow YAML flips **every step's final export** to Parquet — one switch on
`BaseExtractor`, no per-extractor `setup_*_parameters` change. Smaller files, preserved
dtypes, faster reads (meaningful on large multi-year pulls). Error files always stay CSV.

Resolution precedence: **workflow.yml `settings.export_format` > `EDAGRO_EXPORT_FORMAT` env var > default `csv`** (a step-level `export_format:` key overrides the workflow setting for that one step). Parquet needs `pyarrow` (already a dependency).

The cell below runs a tiny Coverage workflow into a temp directory with
`export_format: parquet` and asserts the results file is a readable `.parquet`.

In [ ]:
# Test: parquet export driven purely by the workflow `settings.export_format`.
# Runs a tiny Coverage step into a temp dir and asserts the FINAL results file
# is a readable .parquet (and that no .csv results file was written instead).
import glob
import os
import shutil
import tempfile
from pathlib import Path

import pandas as pd
import yaml

from earthdaily.agriculture.services.workflow_manager import WorkflowManager

tmp_out = Path(tempfile.mkdtemp(prefix="parquet_export_test_"))

# Fresh manager pointed at the temp dir so the assertion sees only this run's files.
pq_manager = WorkflowManager(
    "prod",
    log_to_console=False,
    output_result_dir=str(tmp_out),
    partial_result_dir=str(tmp_out),
)

parquet_wf = {
    "workflow": {
        "name": "parquet-export-test",
        "settings": {
            "max_workers": 5,
            "export_format": "parquet",  # <-- the switch under test
        },
        "steps": [
            {
                "name": "coverage",
                "extractor": "CoverageExtractor",
                "module": "earthdaily.agriculture.extractors.coverage_function",
                "setup": {
                    "method": "setup_coverage_parameters",
                    "params": {"vegetation_index": "NDVI", "start_date": "2025-01-01", "clear_cover_min": 95},
                },
                "run": {
                    # skip_export must be False so the format is actually exercised.
                    "method": "process_entity_coverage_bulk_parallel",
                    "params": {"prefix": "coverage_pq", "skip_export": False},
                },
            }
        ],
    }
}

wf_path = tmp_out / "parquet_export_test.yml"
wf_path.write_text(yaml.dump(parquet_wf, sort_keys=False))
pq_manager.load_workflow(str(wf_path))

# Confirm the setting reached the loaded config (deterministic, no API needed).
assert pq_manager.workflow_cfg["settings"]["export_format"] == "parquet"

pq_results = pq_manager.run_workflow(entity_list=manager.sfd_list.head(3))

parquet_files = glob.glob(str(tmp_out / "coverage_pq_results_*_final.parquet"))
csv_results = glob.glob(str(tmp_out / "coverage_pq_results_*_final.csv"))
n_rows = len(pq_results["coverage"].get("results_df", pd.DataFrame()))

print(f"Coverage rows         : {n_rows}")
print(f"Parquet results files : {[Path(p).name for p in parquet_files]}")
print(f"CSV results files     : {[Path(p).name for p in csv_results]}")

if n_rows > 0:
    assert parquet_files, f"Expected a .parquet results file in {tmp_out}, found: {os.listdir(tmp_out)}"
    assert not csv_results, "Results were written as CSV despite settings.export_format='parquet'"
    back = pd.read_parquet(parquet_files[0])
    assert len(back) == n_rows, (len(back), n_rows)
    print(f"✅ PASS — {len(back)} rows round-tripped from {Path(parquet_files[0]).name}")
else:
    print("⚠️ Coverage returned 0 rows (nothing to export) — rerun with entities that have imagery in the date range.")

shutil.rmtree(tmp_out, ignore_errors=True)